In [6]:
import sys
import joblib
import time
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
sys.path.insert(0, "..")
from scoring_lib.preprocessing import clean_pipeline
from scoring_lib.features import build_features


In [2]:
df = pd.read_csv("cs-training.csv")
df_clean = clean_pipeline(df)
df_final = build_features(df_clean)

print("Dataset final :", df_final.shape)
df_final.columns.tolist()

Dataset final : (149999, 16)


['customer_id',
 'target',
 'revolving_utilization',
 'age',
 'times_30_59_days_late',
 'debt_ratio',
 'open_credit_lines',
 'times_90_days_late',
 'real_estate_loans',
 'times_60_89_days_late',
 'monthly_income_missing_flag',
 'monthly_income_raw',
 'number_of_dependents',
 'total_late_payments',
 'max_delinquency_severity',
 'credit_lines_per_dependent']

In [3]:
# Séparer les features (X) de la cible (y)
X = df_final.drop(columns=["customer_id", "target"])
y = df_final["target"]

# Split stratifié : préserve la même proportion de défauts (6.7%) dans train et test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train :", X_train.shape, "- Taux de défaut :", y_train.mean().round(4))
print("Test  :", X_test.shape, "- Taux de défaut :", y_test.mean().round(4))

Train : (119999, 14) - Taux de défaut : 0.0668
Test  : (30000, 14) - Taux de défaut : 0.0668


In [4]:
# Calcul du ratio de déséquilibre, pour informer les modèles qu'un défaut (classe 1)
# est rare et doit être pondéré plus fortement pendant l'entraînement
ratio_negatif_positif = (y_train == 0).sum() / (y_train == 1).sum()
print("Ratio négatif/positif :", round(ratio_negatif_positif, 2))

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced", n_estimators=100, random_state=42, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        scale_pos_weight=ratio_negatif_positif,
        eval_metric="logloss",
        random_state=42,
    ),
    "LightGBM": LGBMClassifier(
        scale_pos_weight=ratio_negatif_positif,
        random_state=42,
        verbose=-1,
    ),
}

results = []

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start

    y_pred_proba = model.predict_proba(X_test)[:, 1]

    auc_roc = roc_auc_score(y_test, y_pred_proba)
    pr_auc = average_precision_score(y_test, y_pred_proba)

    results.append({
        "Modèle": name,
        "AUC-ROC": round(auc_roc, 4),
        "PR-AUC": round(pr_auc, 4),
        "Temps entraînement (s)": round(training_time, 2),
    })

    print(f"{name} — terminé en {training_time:.2f}s")

results_df = pd.DataFrame(results).sort_values("PR-AUC", ascending=False)
results_df

Ratio négatif/positif : 13.96


c:\Users\soufi\Projects\credit-scoring-platform\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression — terminé en 8.12s
Random Forest — terminé en 2.54s
XGBoost — terminé en 0.58s
LightGBM — terminé en 2.55s


,Modèle,AUC-ROC,PR-AUC,Temps entraînement (s)
3,LightGBM,0.8675,0.3983,2.55
2,XGBoost,0.8524,0.3604,0.58
1,Random Forest,0.8432,0.3416,2.54
0,Logistic Regression,0.8255,0.3202,8.12


In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for name, model in models.items():
    start = time.time()
    scores = cross_val_score(
        model, X_train, y_train,
        cv=cv, scoring="average_precision", n_jobs=-1
    )
    cv_time = time.time() - start

    cv_results.append({
        "Modèle": name,
        "PR-AUC moyenne (CV)": round(scores.mean(), 4),
        "Écart-type (CV)": round(scores.std(), 4),
        "Temps CV (s)": round(cv_time, 2),
    })

    print(f"{name} — PR-AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

cv_results_df = pd.DataFrame(cv_results).sort_values("PR-AUC moyenne (CV)", ascending=False)
cv_results_df

Logistic Regression — PR-AUC: 0.3163 (+/- 0.0132)
Random Forest — PR-AUC: 0.3387 (+/- 0.0106)
XGBoost — PR-AUC: 0.3647 (+/- 0.0056)
LightGBM — PR-AUC: 0.3989 (+/- 0.0075)


,Modèle,PR-AUC moyenne (CV),Écart-type (CV),Temps CV (s)
3,LightGBM,0.3989,0.0075,6.83
2,XGBoost,0.3647,0.0056,4.31
1,Random Forest,0.3387,0.0106,13.26
0,Logistic Regression,0.3163,0.0132,13.01


## Choix du modèle final : LightGBM

Comparaison de 4 modèles sur AUC-ROC, PR-AUC (métrique prioritaire vu le
déséquilibre de classes 93.3%/6.7%), stabilité (validation croisée 5-fold
stratifiée) et temps d'entraînement.

Un split simple 80/20 aurait pu être trompeur (biais possible lié au hasard
du découpage) — la validation croisée à 5 plis confirme la robustesse du
classement : LightGBM obtient la meilleure PR-AUC moyenne (0.3989, écart-type
0.0075), suivi de XGBoost (0.3647, écart-type 0.0056 — le plus stable),
Random Forest (0.3387) et Logistic Regression (0.3163).

XGBoost est marginalement plus stable et significativement plus rapide
(4.31s vs 6.83s en validation croisée), mais l'écart de performance en
faveur de LightGBM (+0.034 de PR-AUC, soit ~9% relatif) est plus déterminant
que ce gain de stabilité/vitesse pour ce cas d'usage — le temps
d'entraînement total reste négligeable en absolu (moins de 10 secondes)
même en production avec ré-entraînement régulier.


In [7]:
lgbm_final = models["LightGBM"]
joblib.dump(lgbm_final, "../scoring_lib/model_lgbm.pkl")

print("Modèle sauvegardé avec succès")

Modèle sauvegardé avec succès
